In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts... or paid for travel/entertainment to Non POs...
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. TARGET CATEGORY CODES: Grabs the full list of flagged Category Codes from the 
      ABAC category_codes_database.
   3. COUPA DATA ONLY: Unions the 7 Coupa files. 
   4. NON-PUBLIC OFFICIAL FILTER: Filters strictly for rows where the `PublicOfficial` 
      column equals 'N' or 'NO'.
   5. ACCOUNT PARSING: Parses the 'Account' string to extract the Cost Center and 
      Category Code.
   6. FILTERING: INNER JOINS the parsed Coupa data to the ABAC Category Codes list.
   7. MAPPING & COUNTING: Maps flagged transactions to AU_IDs using the CC Bootstrap. 
      Rolls up by AU_ID and COUNTS the number of matching transactions.
   8. FINAL OUTPUT: LEFT JOINS this mapped count back to the Master AU anchor to 
      provide a Numerical output (defaults to '0' if no matches).
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Target_Category_Codes AS (
    -- 2. Grab the ABAC category codes to filter for actual T&E expenses
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3. Union all 7 Coupa files into one master dataset
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Non_PO_Transactions AS (
    -- 4 & 5. Parse the Account string and STRICTLY filter for NON-Public Officials
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- Catches 'N' or 'NO'
      AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
),

Flagged_Non_PO_Transactions AS (
    -- 6. Filter the Non-PO transactions against the ABAC Category list
    SELECT 
        c.Cleaned_CC,
        c.CatCode
    FROM Coupa_Non_PO_Transactions c
    INNER JOIN Target_Category_Codes t
        ON c.CatCode = t.CatCode
    WHERE c.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    -- Standardize the CC Mapping table
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 7. Map the flagged Non-PO transactions to their respective AU_IDs and COUNT them
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Non_PO_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 8. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA04' AS QuestionID,               
    -- Numerical Output: Counts the instances. If none, outputs '0'
    COALESCE(CAST(f.Flagged_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 Drill-Down
   
   PURPOSE: Shows EVERY flagged Non-PO transaction from Coupa, regardless of whether 
   the Cost Center maps to an AU, or whether that AU exists in the Master List.
   Useful for tracking unmapped Non-PO T&E expenses.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Target_Category_Codes AS (
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Coupa_Non_PO_Transactions AS (
    -- Extract the Raw String and parsed elements, filtered for PO = No
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        'Coupa' AS Source_System
    FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), TRIM(PublicOfficial), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered WHERE Account LIKE '%-%-%' AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
),

Flagged_Transactions AS (
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.CatCode,
        t.PublicOfficial,
        t.Source_System
    FROM Coupa_Non_PO_Transactions t
    INNER JOIN Target_Category_Codes c
        ON t.CatCode = c.CatCode
    WHERE t.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    f.Cleaned_CC AS Flagged_Cost_Center,
    f.CatCode AS Flagged_Category_Code,
    f.PublicOfficial AS Non_PO_Flag,
    f.Source_System,
    f.Raw_String AS Original_Data_Value
FROM Flagged_Transactions f
LEFT JOIN CC_Mapping m
    ON f.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
ORDER BY BusinessID, f.Cleaned_CC;